## Import drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Code

In [ ]:
# Lệnh này sẽ giải nén project.zip ra thư mục gốc Google Drive
!unzip -o "/content/drive/MyDrive/project.zip" -d "/content/drive/MyDrive/"

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
   creating: /content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/venv/Lib/site-packages/scipy/linalg/tests/__pycache__/
  inflating: /content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/venv/Lib/site-packages/scipy/linalg/tests/__pycache__/test_basic.cpython-38.pyc  
  inflating: /content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/venv/Lib/site-packages/scipy/linalg/tests/__pycache__/test_blas.cpython-38.pyc  
  inflating: /content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/venv/Lib/site-packages/scipy/linalg/tests/__pycache__/test_cythonized_array_utils.cpython-38.pyc  
  inflating: /content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/venv/Lib/site-packages/scipy/linalg/tests/__pycache__/test_cython_blas.cpython-38.pyc  
  inflating: /content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/venv/Lib/site-packages/scipy/linalg/tests/__pycache_

In [2]:
%cd "/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition"

/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition


In [ ]:
!rm -rf results/ data/processed/ data/external/

## Install lib

In [ ]:
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 27.6 MB/s eta 0:00:00


## Tier 1

In [ ]:
# Tạo sơ đồ danh mục hình ảnh (stimulus categories)
!python -m src.utils.generate_category_map

# Chạy tiền xử lý dữ liệu thô (.xlsx -> Parquet sạch)
!python -m src.tier1_preprocessing.preprocess

Successfully generated category mapping for 100 images at data/metadata/stimulus_categories.csv
--- Tier 1: Loading raw data ---
Loading Fixations: 100% 160/160 [00:17<00:00,  9.06it/s]
Loading Fixations: 100% 48/48 [00:37<00:00,  1.27it/s]
Total raw fixations loaded: 293740
Spatial boundary filter: removed 5037 out of 293740 fixations (1.71%)
Temporal duration filter: removed 7666 out of 288703 fixations (2.66%)
Successfully saved 281037 fixations to data/processed/clean_fixations.parquet
--- Tier 1 Preprocessing Completed Successfully ---


## Tier 2

In [ ]:
# Trích xuất đặc trưng mức độ thử nghiệm (stimulus-level)
!python -m src.tier2_features.stimulus_features

# Tổng hợp và tính các đặc trưng Delta mức độ đối tượng (subject-level)
!python -m src.tier2_features.subject_aggregator

Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Extracting features per trial...
100% 20695/20695 [00:23<00:00, 884.36it/s]
Extracted features for 20695 trials. Saved to data/processed/features_stimulus_level.csv
Loading stimulus-level features from data/processed/features_stimulus_level.csv...
Loading category mapping from data/metadata/stimulus_categories.csv...
Computing mean feature values per subject, per category...
Computing contextual delta features...
Aggregated subject-level features for 208 subjects. Saved to data/processed/features_subject_level.csv


## Tier 3

In [ ]:
# Huấn luyện mô hình XGBoost, lưu kết quả Out-Of-Fold (OOF) của Tier 3
!python -m src.tier3_tabular.tabular_models --model xgboost

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:175: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [16:12:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iter

## Tier 4

In [ ]:
# Trích xuất đặc trưng hình ảnh bằng ResNet50 thật
!python -m src.utils.extract_resnet_features

# Xây dựng đồ thị từ đặc trưng ảnh thật
!python -m src.tier4_advanced.graph_builder

!python scripts/train_tier4.py

 VISUAL FEATURE EXTRACTION (ResNet50 Baseline)
Found 100 stimulus images in EMS/Images.
Loading pre-trained ResNet50 on device: cuda...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 177MB/s]
Extracting feature maps for all stimulus images...
Extracting image features: 100% 100/100 [01:12<00:00,  1.39it/s]
Mapping visual features to subject fixations...
Mapping to trials: 100% 16716/16716 [00:56<00:00, 298.17it/s]

Successfully extracted ResNet50 features. Saved to data/external/feature_dict_RINet.npy
Total trials mapped: 16716
Feature vector dimension: 2048
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Global pupil stats (minor leakage — see NOTE above): Mean=1218.29, Std=614.17
Loading RINet features from data/external/feature_dict_RINet.npy...
Building spatiotemporal graphs...
100% 16716/16716 [00:37<00:00, 442.70it/s]
Successfully cons

In [ ]:
!python scripts/train_tier4.py

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Found existing checkpoint at results/cefam/checkpoints/cefam_fold_0_best.pt. Evaluating...
Loaded checkpoint — Val Subject AUC: 0.9722

==================== 

## Tier 5

In [ ]:
# Kết hợp OOF dự đoán của Tier 3 (XGBoost) và Tier 4 (GNN) để đưa ra dự đoán tối ưu
!python scripts/run_tier5.py --plot --calibrate

2026-06-26 16:57:08.063705: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-26 16:57:08.134310: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.007, 0.994]
[Tier5] Test prediction files not found; skipping test inference.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9416  ACC: 0.8

## STGNN

In [ ]:
!rm -rf results/stgnn/

In [ ]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.1360 TrAUC=0.7294 | VaLoss=0.6894 VaTrialAUC=0.6447 VaSubjAUC=0.6364
Ep 005/200 | TrLoss=0.0726 TrAUC=0.7087 | VaLoss=0.1481 VaTrialAUC=0.5296 VaSubjAUC=0.6111
Ep 010/200 | TrLoss=0.0631 TrAUC=0.7379 | VaLoss=0.6962 VaTrialAUC=0.6619 VaSubjAUC=0.7020
Ep 015/200 | TrLoss=0.0628 TrAUC=0.7484 | VaLoss=0.5215 VaTrialAUC=0.6273 VaSubjAUC=0.7146
Ep 020/200 | TrLoss=0.062

In [ ]:
!python scripts/calibrate_threshold.py --preds results/stgnn/stgnn_subject_val_predictions.csv

Loading predictions from results/stgnn/stgnn_subject_val_predictions.csv...
Overall Subject-Level AUC: 0.7361

--- Metrics at Default Threshold (0.5000) ---
  Accuracy: 0.5000
  F1-Score: 0.0000
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
  Precision: 0.0000
  Recall: 0.0000

--- Optimized Metrics at Calibrated Threshold (0.3989) ---
  Accuracy: 0.6625
  F1-Score: 0.7404
  Precision: 0.6016
  Recall (Sensitivity): 0.9625
  Specificity: 0.3625


## BICA

In [3]:
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --config configs/bica_config.yaml

!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/evaluate_bica.py" --config configs/bica_config.yaml

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0505 | Train AUC: 0.8948 | Val Loss: 0.0744 | Val Trial AUC: 0.8001 | Val Subject AUC: 0.8384
Epoch 005/150 | Train Loss: 0.0400 | Train AUC: 0.9373 | Val Loss: 0.0741 | Val Trial AUC: 0.8402 | Val Subject AUC: 0.8914
Epoch 010/150 | Train Loss: 0.0389 | Train AUC: 0.9414 | Val Loss: 0.0759 | Val Trial AUC: 0.8632 | Val Sub

## Ablation

In [4]:
# 1. Xóa các file phân tích cũ
!rm -rf experiments/ablation/results/ experiments/ablation/figures/

# 2. Chạy phân tích so sánh và độ nhạy
!python experiments/ablation/run_ablation_analysis.py

# 3. Vẽ biểu đồ so sánh xuất bản chất lượng cao
!python experiments/ablation/plot_ablation.py

ABLATION STUDY - Eye Movement-Based Schizophrenia Recognition

Loading model predictions...
Loaded 4 models: ['GNN+CEFAM (Full Hybrid)', 'ST-GNN (GNN Only)', 'BiCA-HS (Transformer)', 'XGBoost (Tabular Only)']

F1: FULL MODEL COMPARISON (Main Ablation Table)

--- GNN+CEFAM (Full Hybrid) ---
  AUC-ROC:  0.9416
  ACC @0.5: 0.8812
  F1  @0.5: 0.8790
  ACC @opt: 0.8875 (th=0.3301)
  F1  @opt: 0.8902
  Sens@opt: 0.9125
  Spec@opt: 0.8625

--- ST-GNN (GNN Only) ---
  AUC-ROC:  0.7361
  ACC @0.5: 0.5000
  F1  @0.5: 0.0000
  ACC @opt: 0.6625 (th=0.3989)
  F1  @opt: 0.7404
  Sens@opt: 0.9625
  Spec@opt: 0.3625

--- BiCA-HS (Transformer) ---
  AUC-ROC:  0.9525
  ACC @0.5: 0.8875
  F1  @0.5: 0.8953
  ACC @opt: 0.9062 (th=0.5187)
  F1  @opt: 0.9091
  Sens@opt: 0.9375
  Spec@opt: 0.8750

--- XGBoost (Tabular Only) ---
  AUC-ROC:  0.8702
  ACC @0.5: 0.7812
  F1  @0.5: 0.7853
  ACC @opt: 0.8063 (th=0.5953)
  F1  @opt: 0.8050
  Sens@opt: 0.8000
  Spec@opt: 0.8125

F2: COMPONENT CONTRIBUTION ANALYSIS

 

In [5]:
# Huấn luyện và đánh giá Tier 5 sử dụng kết quả OOF của BiCA-HS
!python scripts/run_tier5.py \
    --tier4-oof results/bica/bica_subject_val_predictions.csv \
    --plot --calibrate

2026-06-27 05:23:27.124532: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 05:23:27.190576: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.000, 0.967]
[Tier5] Test prediction files not found; skipping test inference.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9525  ACC: 0.8

In [3]:
!python scratch_feature_importance.py

  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [05:33:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [05:33:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, i

In [3]:
!python scratch_delong_test.py

 DELONG SIGNIFICANCE TEST: BiCA-HS vs Tier 5 Ensembles
Number of subjects : 160
Tier 4 (BiCA-HS) AUC : 0.9525
------------------------------------------------------------
 1. CROSS-VALIDATED ENSEMBLE (Generalization performance)
------------------------------------------------------------
Tier 5 (CV Meta) AUC : 0.9502
Z-statistic          : 0.2054
P-value              : 0.837289
Result is NOT statistically significant at alpha=0.05 (p >= 0.05).
------------------------------------------------------------
 2. FULL META-LEARNER ENSEMBLE (Final model deployment)
------------------------------------------------------------
Tier 5 (Full Meta) AUC: 0.9619
Z-statistic           : -1.0286
P-value               : 0.303664
Result is NOT statistically significant at alpha=0.05 (p >= 0.05).
The difference in performance could be due to chance.
